# PLAYSTATS: Basketball Player Statistics

This notebook fetches and displays tabular basketball player statistics from the Basketball Stats Vlaanderen website for a specific player and season.

In [1]:
# Import Required Libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [23]:
# Fetch Player Data from Webpage
url = "https://vblstats.wisseq.eu/speler/BVBL744354"
response = requests.get(url)
html_content = response.text

In [24]:
html_content

'<!DOCTYPE html>\r\n<html lang="en" data-critters-container>\r\n<head>\r\n  <meta charset="utf-8">\r\n  <title>Vblstats</title>\r\n  <base href="/">\r\n  <meta name="viewport" content="width=device-width, initial-scale=1">\r\n  <link rel="icon" type="image/x-icon" href="./assets/favicon.ico">\r\n  <link rel="preconnect" href="https://fonts.gstatic.com">\r\n  <!-- <link\r\n    href="https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;500&display=swap"\r\n    rel="stylesheet"\r\n  /> -->\r\n  <style>@font-face{font-family:\'Material Icons\';font-style:normal;font-weight:400;font-display:swap;src:url(https://fonts.gstatic.com/s/materialicons/v143/flUhRq6tzZclQEJ-Vdg-IuiaDsNc.woff2) format(\'woff2\');}.material-icons{font-family:\'Material Icons\';font-weight:normal;font-style:normal;font-size:24px;line-height:1;letter-spacing:normal;text-transform:none;display:inline-block;white-space:nowrap;word-wrap:normal;direction:ltr;-webkit-font-feature-settings:\'liga\';-webkit-font-smooth

In [13]:
# Fetch data from the updated URL with player stats from last year
url = "https://vblstats.wisseq.eu/speler/BVBL744354%25"
response = requests.get(url)
html_content = response.text

soup = BeautifulSoup(html_content, 'html.parser')

# Find the player stats table (usually the one with team data)
all_tables = soup.find_all('table')
stats_df = None

for i, table in enumerate(all_tables):
    try:
        temp_df = pd.read_html(str(table))[0]
        # Check if this looks like the player stats table (has team column)
        if any('Team' in str(col) for col in temp_df.columns) or any('ploeg' in str(col).lower() for col in temp_df.columns):
            stats_df = temp_df
            break
    except:
        continue

if stats_df is not None:
    # Clean up the column names and select only required ones
    # First, print the actual columns to help with mapping
    print("Original columns:", stats_df.columns.tolist())
    
    # Rename/map columns based on the actual column names from the table
    # This is a placeholder - we'll need to adjust based on the actual columns
    column_mapping = {
        'Ploeg': 'Team',
        'Reeks': 'Reeks',
        'Wed': 'Wed',
        '1p': 'pt1',
        '2p': 'pt2',
        '3p': 'pt3',
        'Tot': 'Tot',
        'Gem': 'avgPts'
    }
    
    # Apply renaming where possible and select columns
    stats_df = stats_df.rename(columns=column_mapping)
    
    # Select only the columns we need (if they exist)
    available_columns = [col for col in column_mapping.values() if col in stats_df.columns]
    final_df = stats_df[available_columns]
    
    display(final_df)
else:
    print("Could not find the player stats table on the page.")

Could not find the player stats table on the page.


In [14]:
# Extract player stats table with the specific column structure
url = "https://vblstats.wisseq.eu/speler/BVBL744354%25"
response = requests.get(url)
html_content = response.text

soup = BeautifulSoup(html_content, 'html.parser')

# Function to extract team name from complex nested structure
def extract_team_name(cell):
    team_link = cell.find('a')
    if team_link:
        return team_link.text.strip()
    return None

# Create empty lists to store data
teams = []
reeks_list = []
games = []
pt1_list = []
pt2_list = []
pt3_list = []
tot_list = []
avg_list = []

# Find all rows in the table (they have mat-row class in Angular Material tables)
rows = soup.find_all('tr', class_='mat-mdc-row')

for row in rows:
    # Get all cells in the row
    cells = row.find_all('td', class_='mat-mdc-cell')
    
    if len(cells) >= 8:  # Ensure we have enough cells
        # Extract team from the complex nested structure
        team_cell = cells[0]
        team_name = extract_team_name(team_cell)
        if not team_name and team_cell.find('tr'):
            team_name = extract_team_name(team_cell.find('tr'))
        teams.append(team_name if team_name else "Unknown")
        
        # Extract other columns (they're simpler)
        reeks_cell = cells[1].find('a')
        reeks_list.append(reeks_cell.text.strip() if reeks_cell else cells[1].text.strip())
        
        games.append(cells[2].text.strip())
        pt1_list.append(cells[3].text.strip())
        pt2_list.append(cells[4].text.strip())
        pt3_list.append(cells[5].text.strip())
        tot_list.append(cells[6].text.strip())
        avg_list.append(cells[7].text.strip())

# Create DataFrame with the specified column names
player_stats_df = pd.DataFrame({
    'Team': teams,
    'Reeks': reeks_list,
    'Wed': games,
    'pt1': pt1_list,
    'pt2': pt2_list,
    'pt3': pt3_list,
    'Tot': tot_list,
    'avgPts': avg_list
})

# Convert numeric columns
numeric_cols = ['Wed', 'pt1', 'pt2', 'pt3', 'Tot', 'avgPts']
for col in numeric_cols:
    player_stats_df[col] = pd.to_numeric(player_stats_df[col], errors='coerce')

# Display the clean DataFrame
display(player_stats_df)

,Team,Reeks,Wed,pt1,pt2,pt3,Tot,avgPts


In [15]:
# Extract player stats from Angular Material table
url = "https://vblstats.wisseq.eu/speler/BVBL744354%25"
response = requests.get(url)
html_content = response.text

soup = BeautifulSoup(html_content, 'html.parser')

# Debug - count rows to verify we're finding them
rows = soup.find_all('tr', class_='mat-mdc-row')
print(f"Found {len(rows)} rows with class 'mat-mdc-row'")

# Try alternative class patterns
if len(rows) == 0:
    rows = soup.find_all('tr', attrs={'mat-row': ''})
    print(f"Found {len(rows)} rows with attribute 'mat-row'")

# Function to extract team name from nested structure
def extract_team_name(cell):
    links = cell.find_all('a')
    for link in links:
        if link.text.strip():
            return link.text.strip()
    return "Unknown"

# Create empty lists
teams = []
reeks_list = []
games = []
pt1_list = []
pt2_list = []
pt3_list = []
tot_list = []
avg_list = []

# Process each row in the table
for row in soup.select('tr.mat-mdc-row'):
    cells = row.select('td.mat-mdc-cell')
    
    if len(cells) >= 8:
        # Extract team name from complex structure
        team_cell = cells[0]
        team_name = extract_team_name(team_cell)
        teams.append(team_name)
        
        # Extract other data
        reeks = extract_team_name(cells[1])
        reeks_list.append(reeks)
        
        games.append(cells[2].text.strip())
        pt1_list.append(cells[3].text.strip())
        pt2_list.append(cells[4].text.strip())
        pt3_list.append(cells[5].text.strip())
        tot_list.append(cells[6].text.strip())
        avg_list.append(cells[7].text.strip())

# Create DataFrame
player_stats_df = pd.DataFrame({
    'Team': teams,
    'Reeks': reeks_list,
    'Wed': games,
    'pt1': pt1_list,
    'pt2': pt2_list,
    'pt3': pt3_list, 
    'Tot': tot_list,
    'avgPts': avg_list
})

# Convert numeric columns
numeric_cols = ['Wed', 'pt1', 'pt2', 'pt3', 'Tot', 'avgPts']
for col in numeric_cols:
    player_stats_df[col] = pd.to_numeric(player_stats_df[col], errors='coerce')

# Display the DataFrame
display(player_stats_df)

Found 0 rows with class 'mat-mdc-row'
Found 0 rows with attribute 'mat-row'


,Team,Reeks,Wed,pt1,pt2,pt3,Tot,avgPts


In [12]:
html_content

'<!DOCTYPE html>\r\n<html lang="en" data-critters-container>\r\n<head>\r\n  <meta charset="utf-8">\r\n  <title>Vblstats</title>\r\n  <base href="/">\r\n  <meta name="viewport" content="width=device-width, initial-scale=1">\r\n  <link rel="icon" type="image/x-icon" href="./assets/favicon.ico">\r\n  <link rel="preconnect" href="https://fonts.gstatic.com">\r\n  <!-- <link\r\n    href="https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;500&display=swap"\r\n    rel="stylesheet"\r\n  /> -->\r\n  <style>@font-face{font-family:\'Material Icons\';font-style:normal;font-weight:400;font-display:swap;src:url(https://fonts.gstatic.com/s/materialicons/v143/flUhRq6tzZclQEJ-Vdg-IuiaDsNc.woff2) format(\'woff2\');}.material-icons{font-family:\'Material Icons\';font-weight:normal;font-style:normal;font-size:24px;line-height:1;letter-spacing:normal;text-transform:none;display:inline-block;white-space:nowrap;word-wrap:normal;direction:ltr;-webkit-font-feature-settings:\'liga\';-webkit-font-smooth

In [11]:
# Extract the first table from the HTML content and load it into a DataFrame
import pandas as pd
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_content, 'html.parser')
tables = soup.find_all('table')

if tables:
    table_html = str(tables[0])
    df = pd.read_html(table_html)[0]
    display(df)
else:
    print("No table found on the page.")

No table found on the page.


In [20]:
def get_player_stats(player_id, base_url="https://vblstats.wisseq.eu/speler/"):
    
    import requests
    from bs4 import BeautifulSoup
    import pandas as pd
    
    # Construct URL and fetch content
    url = base_url + player_id
    response = requests.get(url)
    html_content = response.text
    
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find all rows in the Angular Material table
    rows = soup.find_all('tr', class_='mat-mdc-row')
    
    # Try alternative class patterns if needed
    if len(rows) == 0:
        rows = soup.find_all('tr', attrs={'mat-row': ''})
    
    # Create empty lists to store data
    teams = []
    reeks_list = []
    games = []
    pt1_list = []
    pt2_list = []
    pt3_list = []
    tot_list = []
    avg_list = []
    
    # Process each row in the table
    for row in soup.select('tr.mat-mdc-row'):
        cells = row.select('td.mat-mdc-cell')
        
        if len(cells) >= 8:
            # Extract team name
            team_cell = cells[0]
            team_name = extract_team_name(team_cell)
            teams.append(team_name)
            
            # Extract other data
            reeks = extract_team_name(cells[1])
            reeks_list.append(reeks)
            
            games.append(cells[2].text.strip())
            pt1_list.append(cells[3].text.strip())
            pt2_list.append(cells[4].text.strip())
            pt3_list.append(cells[5].text.strip())
            tot_list.append(cells[6].text.strip())
            avg_list.append(cells[7].text.strip())
    
    # Create DataFrame
    player_stats_df = pd.DataFrame({
        'Team': teams,
        'Reeks': reeks_list,
        'Wed': games,
        'pt1': pt1_list,
        'pt2': pt2_list,
        'pt3': pt3_list, 
        'Tot': tot_list,
        'avgPts': avg_list
    })
    
    # Convert numeric columns
    numeric_cols = ['Wed', 'pt1', 'pt2', 'pt3', 'Tot', 'avgPts']
    for col in numeric_cols:
        player_stats_df[col] = pd.to_numeric(player_stats_df[col], errors='coerce')
    
    return player_stats_df

# def extract_team_name(cell):
  

# Example usage:player_stats = get_player_stats("BVBL744354%25")
# display(player_stats)

In [21]:
a = get_player_stats("BVBL744354%25")

In [22]:
a

,Team,Reeks,Wed,pt1,pt2,pt3,Tot,avgPts


In [3]:
html_content

'<!DOCTYPE html>\r\n<html lang="en" data-critters-container>\r\n<head>\r\n  <meta charset="utf-8">\r\n  <title>Vblstats</title>\r\n  <base href="/">\r\n  <meta name="viewport" content="width=device-width, initial-scale=1">\r\n  <link rel="icon" type="image/x-icon" href="./assets/favicon.ico">\r\n  <link rel="preconnect" href="https://fonts.gstatic.com">\r\n  <!-- <link\r\n    href="https://fonts.googleapis.com/css2?family=Roboto:wght@300;400;500&display=swap"\r\n    rel="stylesheet"\r\n  /> -->\r\n  <style>@font-face{font-family:\'Material Icons\';font-style:normal;font-weight:400;font-display:swap;src:url(https://fonts.gstatic.com/s/materialicons/v143/flUhRq6tzZclQEJ-Vdg-IuiaDsNc.woff2) format(\'woff2\');}.material-icons{font-family:\'Material Icons\';font-weight:normal;font-style:normal;font-size:24px;line-height:1;letter-spacing:normal;text-transform:none;display:inline-block;white-space:nowrap;word-wrap:normal;direction:ltr;-webkit-font-feature-settings:\'liga\';-webkit-font-smooth

In [4]:
# Extract and print only the relevant stats row from the 'competitie' table
soup = BeautifulSoup(html_content, 'html.parser')

# Find the table with 'competitie' in the caption or header
competitie_table = None
for table in soup.find_all('table'):
    caption = table.find('caption')
    if caption and 'competitie' in caption.text.lower():
        competitie_table = table
        break
    # Fallback: check for header row containing 'competitie'
    thead = table.find('thead')
    if thead and 'competitie' in thead.text.lower():
        competitie_table = table
        break

if competitie_table:
    rows = competitie_table.find_all('tr')
    for row in rows:
        cells = [cell.get_text(strip=True) for cell in row.find_all(['th', 'td'])]
        if cells and cells[0].startswith('Alle wedstrijden'):
            print('\t'.join(cells))
            break
else:
    print("Competitie table not found.")

Competitie table not found.


In [5]:
competitie_table

In [4]:
# Parse and Extract Tabular Data
soup = BeautifulSoup(html_content, 'html.parser')
tables = soup.find_all('table')

# For demonstration, extract the first table (adjust as needed)
if tables:
    table_html = str(tables[0])
else:
    table_html = None

In [6]:
# Display Data in a Pandas DataFrame
if table_html:
    df = pd.read_html(table_html)[0]
    display(df)
else:
    print("No table found on the page.")

NameError: name 'table_html' is not defined

In [7]:
# Basic Data Analysis (e.g., Averages, Totals)
if table_html:
    # Example: Show column names and basic stats
    print("Columns:", df.columns.tolist())
    print("\nSummary statistics:")
    display(df.describe(include='all'))
else:
    print("No data to analyze.")

Columns: ['Competitie', 'Punten', 'F']

Summary statistics:


,Competitie,Punten,F
count,0,0,0
unique,0,0,0
top,NaN,NaN,NaN
freq,NaN,NaN,NaN
